# GeoGuide: Geometryczne Sterowanie Trajektorią w Modelach Dyfuzji (Geometric guidance of diffusion models)

Modele dyfuzyjne świetnie radzą sobie z generowaniem obrazów na podstawie danych, na których były trenowane. Jednak sterowanie pre-trenowanym modelem dyfuzyjnym w celu wygenerowania elementów z wcześniej nieznanych klas jest znacznie trudniejsze. 
Powszechne podejście wykorzystujące probabilistyczne wprowadzanie klasyfikatora (Classifier Guidance, ADM-G) dostarcza znaczącą lukę jakości (zmniejszenie wyniku FID) między generowaniem pod kontrolą sterowania od zera, a modelem warunkowanym w trakcie treningu.

Dzieje się tak, bo ADM-G dostarcza bardzo słabą sterowność pod koniec procesu odszumiania. Normy wyliczonego gradientu zmniejszają się, przez co model traci detale i cechy charakterystyczne wybranej klasy.

**GeoGuide** rozwiązuje ten problem. Zamienia on perspektywę czysto probabilistycznego podejścia na rozwiązanie *metryczne*. Zamiast skalować gradient wartością, zanikającą pod koniec trajektorii metoda normalizuje poprawkę do stałej długości, w skutek czego proces odszumiania jest zbliżony do oryginalnej rozmaitości danych (data manifold) przez cały czas, a nie tylko na początku.

![GeoGuide comparison](img/geoguide_comp.png)

In [1]:
!git clone https://github.com/mateuszpoleski/geoguide
%cd geoguide

Cloning into 'geoguide'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 62 (delta 8), reused 57 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 1013.74 KiB | 5.57 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/home/bartosz/Desktop/Studia/RepresentationDL/trajectory-guidence/geoguide


## Jak działa GeoGuide?

GeoGuide normalizuje i skaluje gradient w procesie odszumiania (backward diffusion denoising process), by dostarczyć poprawki ukierunkowane w pożądaną klasę wektorem o równej odległości i normie na całej trajektorii.

### Klasyczny ADM-G (Baseline)

W podejściu Classifier Guidance (probabilistycznym), proces odszumiania modyfikuje wektor szumu dodając do niego zeskalowany względem kroku czasowego gradient z dotrenowanego w locie klasyfikatora (bądź jego straty):
$$ A_t = \gamma_t \nabla \log p(y|x_t) $$
Norma tej modyfikacji spada pod koniec trajektorii odszumiania, bo predykcja rozkładu klasy dla powoli wyostrzającego i wyłaniającego się powoli obrazu drastycznie słabnie.

### Geometryczne Sterowanie Trajektorii (GeoGuide)

Zamiast modyfikować trajektorię odszumiania za pomocą surowego, wygasającego gradientu, GeoGuide traktuje nawigację w przestrzeni danych jako problem geometryczny. Metoda ta normalizuje wektor gradientu, dzięki czemu na każdym kroku odszumiania wprowadzana jest poprawka o stałej, przewidywalnej długości (tzw. *fixed-length updates*). Znormalizowany wektor sterujący wyznaczany jest ze wzoru:
$$ A_t = \sqrt{\frac{D}{T}} \frac{\nabla \log p(y|x_t)}{||\nabla \log p(y|x_t)||} $$

Zaktualizowany stan $x_{t-1}$ powstaje poprzez dodanie tej znormalizowanej poprawki do standardowego kroku dyfuzji, przeskalowanej dodatkowo przez wagę sterowania $s$:
$$ x_{t-1} = \mu(t, x_t) + \sqrt{\gamma_t} \epsilon_t + s A_t $$

![GeoGuide guidance after 30](img/geoguide_guide.png)

gdzie:
- $D$ - wymiarowość przestrzeni danych (obrazu).
- $T$ - całkowita liczba kroków procesu odszumiania.

---

### Różnice w algorytmach: ADM-G vs GeoGuide

Proces odszumiania w obu przypadkach przebiega podobnie. Główna różnica sprowadza się do sposobu obliczania wektora sterującego $A_t$. W klasycznym ADM-G zależy on ściśle od wariancji $\gamma_t$ i normy gradientu, która w trakcie dyfuzji bywa niestabilna i mocno gaśnie, co osłabia działanie klasyfikatora. GeoGuide narzuca normalizację wymuszając stałą długość kroku poprawki na iterację, co stabilizuje trajektorię i uodparnia generację na problem gasnących gradientów.

**Algorytm 1: ADM-G (Klasyczne sterowanie prawdopodobieństwem)**
```
Wejście: etykieta y, siła sterowania s
x_T ← próbkuj z N(0, I)
Dla t = T w dół do 1:
    eps_t ← próbkuj z N(0, I)

    # Podejście probabilistyczne (wektor sterujący zależny od normy gradientu log-prawdopodobieństwa)
    A_t = γ_t · ∇_x log p(y | x_t)
    x_{t-1} = μ(t, x_t) + √γ_t · eps_t + s · A_t
Zwróć x_0
```

**Algorytm 2: GeoGuide (Geometryczne sterowanie trajektorią)**
```
Wejście: etykieta y, siła sterowania s
x_T ← próbkuj z N(0, I)
Dla t = T w dół do 1:
    eps_t ← próbkuj z N(0, I)
    
    # Podejście geometryczne (znormalizowany wektor poprawki o stałej długości)
    A_t = √(D/T) · ( ∇_x log p(y | x_t) / ‖∇_x log p(y | x_t)‖ )
    x_{t-1} = μ(t, x_t) + √γ_t · eps_t + s · A_t
Zwróć x_0
```

![Gradient normalization impact](img/geoguide_main_idea.png)

---

## Kluczowe Cechy Metodologii GeoGuide

1. **Aktualizacje o Stałej Długości (Fixed-Length Updates):**
Dzięki znormalizowanej stałej normie wektora sterującego problem znikającego gradientu (charakterystyczny dla końcowych etapów dyfuzji w ADM-G) zostaje wyeliminowany. Sprawia to, że proces ciągle dysponuje siłą sterującą, pozwalając na skuteczne uwzględnianie i dopracowywanie drobnych detali oraz tekstur w późnych etapach trajektorii generowania obrazu.

2. **Łatwość Integracji (Plug-and-Play):**
GeoGuide działa wyłącznie poprzez modyfikację na etapie próbkowania (sampling). Metoda jest w pełni niezależna od wag – nie wymaga absolutnie żadnej ingerencji w architekturę ani dotrenowywania bazowego modelu dyfuzyjnego czy używanego klasyfikatora. 

3. **Znacząca Poprawa Jakości Generacji (FID Score):**
Zastąpienie klasycznego, niestabilnego gradientu stałą aktualizacją geometryczną ($A_t$) przekłada się na dużo wyższą wierność generowanych detali względem podanych warunków. Eksperymenty jednoznacznie dowodzą, że GeoGuide pozwala na drastyczne obniżenie metryki FID (Fréchet Inception Distance), rekompensując braki jakościowe generowane przez standardowe metody probabilistyczne.

## Zadanie – Matematyczny krok kierowania dyfuzją w GeoGuide

GeoGuide opiera się na naprawie zanikających poprawek dyfuzyjnych poprzez wprowadzanie sztywnych poprawek metrycznych do tensora szumu na podstawie geometrii problemu - $D$ oraz $T$.

Twoim zadaniem jest uzupełnienie prostej funkcji kroku procesu w tył `p_sample_backward`, w której ukierunkowanie wektora oznaczono `???`.

**Trzy rzeczy do zrobienia:**
1. Policz normę ($L_2$) obciążenia uzyskanego gradientu próbującego stworzyć nam klasę - $g$.
2. Oblicz znormalizowaną i uśrednioną normami popraweczkę $A_t$ zgodnie ze wzorem $\sqrt{D/T} \frac{g}{||g||} $.
3. Uzupełnij aktualizację kroku $x_{t-1}$ o matematyczny kierunkowy wektor wpędzający $s A_t$.

In [ ]:
import torch
import math

class DummyClassifier(torch.nn.Module):
    def forward(self, x):
        # Symulowany log p(y|x_t)
        return torch.sum(x ** 2)

class GeoGuideDiffusionModel:
    def __init__(self, D, T, scale=1.0):
        self.D = D  # Wymiarowość (Dimension)
        self.T = T  # Liczba wszystkich kroków (Time steps)
        self.s = scale  # Skala sterowania klasyfikatorem (gradient scale s)
        self.classifier = DummyClassifier()

    def get_classifier_gradient(self, x_t):
        x_t_temp = x_t.clone().detach().requires_grad_(True)
        log_p = self.classifier(x_t_temp)
        log_p.backward()
        return x_t_temp.grad.detach()

    def p_sample_backward(self, x_t, mu_t_x_t, gamma_t, eps_t):
        # Predykcja straty klasy - wektor uderzeń g 
        g = self.get_classifier_gradient(x_t)
        
        # 1. Norma wektora gradientu (uważaj na zera przy dzieleniu, np. + 1e-8)
        g_norm = ???

        # 2. Obliczenie zmiennej A_t używając wektora poprawek geometrycznych wzoru na stałą
        norm_factor = math.sqrt(self.D / self.T)
        A_t = ???

        # 3. Krok aktualizacji: x_{t-1} = mu(t, x_t) + gamma_t * eps_t + s * A_t
        x_t_minus_1 = ???

        return x_t_minus_1.detach()

# --- Test ---
D_dim = 1 * 3 * 64 * 64
T_steps = 1000

geoguide_step = GeoGuideDiffusionModel(D=D_dim, T=T_steps, scale=2.5)

x_curr = torch.randn(1, 3, 64, 64)
mu_val = torch.randn_like(x_curr)
gamma_val = 0.5
eps_val = torch.randn_like(x_curr)

x_next = geoguide_step.p_sample_backward(x_curr, mu_val, gamma_val, eps_val)

assert x_next.shape == (1, 3, 64, 64), "Zły kształt tensora!"
assert not torch.isnan(x_next).any(), "Tensor zawiera NaN!"
print("Krok GeoGuide wprowadzony z użyciem stałego rozmiaru korekty geometrycznej wykonany poprawnie! Wymiary próbki:", x_next.shape)